# Figure 4: Hard Generalization On Norman

- Figure / panels: `Fig4a`, `Fig4b`, `Fig4c`, `Fig4d`, `Fig4e`, `Fig4f` are the current main-text/supp outputs; hard-regime focus is supplementary reserve
- Inputs: resolved Norman metrics files from `artifacts/results/norman/...`, `artifacts/results/<baseline>/norman/metrics.csv`, Norman payload `pkl`
- Outputs: `artifacts/paper_figures/main/Fig4_NormanGeneralization/*`
- Run order: run after Fig2; this notebook is the Norman subgroup and unseen-combo module
- Dataset selector: edit `DATASET` in the parameter cell below; this notebook currently supports `norman` only
- Role: Main-text stress-test module with unified thin-bar styling; the optional hard-regime focus panel remains off by default
- Baseline note: `scGPT` is included as a foundation-model baseline, not as a task-specific perturbation model


In [1]:
from __future__ import annotations

import importlib
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from scripts.common.paper_plot_style import apply_gears_paper_style, style_axis
from scripts.common.split_utils import split_by_dataset_policy
from scripts.trishift.analysis import baseline_panel

apply_gears_paper_style(font_scale=1.0)
from scripts.trishift.analysis._result_adapter import load_metrics_df, load_payload_item, resolve_result
from scripts.systema._core import baselines_core as systema_core
from trishift._utils import normalize_condition
importlib.reload(baseline_panel)


<module 'scripts.trishift.analysis.baseline_panel' from 'E:\\CODE\\trishift\\scripts\\trishift\\analysis\\baseline_panel.py'>

In [2]:
AVAILABLE_DATASETS = ['norman']
DATASET = 'norman'  # edit here
if DATASET not in AVAILABLE_DATASETS:
    raise ValueError(f'Unsupported dataset: {DATASET}. Available: {AVAILABLE_DATASETS}')

MODELS = ['trishift_nearest', 'biolord', 'gears', 'genepert', 'scgpt', 'systema_nonctl_mean', 'systema_matching_mean']  # edit here; scgpt is the foundation-model baseline
AVAILABLE_SPLIT_IDS = [1, 2, 3, 4, 5]
SELECTED_SPLIT_IDS = AVAILABLE_SPLIT_IDS.copy()  # edit here
EXPORT_FIG4C = False  # edit here; optional diagnostic export not used in the manuscript panel set
invalid_split_ids = [split_id for split_id in SELECTED_SPLIT_IDS if split_id not in AVAILABLE_SPLIT_IDS]
if invalid_split_ids:
    raise ValueError(f'Unknown split ids: {invalid_split_ids}. Available: {AVAILABLE_SPLIT_IDS}')
SPLIT_IDS = [int(split_id) for split_id in SELECTED_SPLIT_IDS]
OUT_ROOT = repo_root / 'artifacts' / 'paper_figures' / 'main' / 'Fig4_NormanGeneralization'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_LABELS = {'trishift_nearest': 'TriShift', 'biolord': 'biolord', 'gears': 'GEARS', 'genepert': 'GenePert', 'scgpt': 'scGPT', 'systema_nonctl_mean': 'Perturbed mean', 'systema_matching_mean': 'Matching mean', 'Truth': 'Truth'}
MODEL_COLORS = {'TriShift': '#9FD9D3', 'biolord': '#F0806A', 'GEARS': '#F2B56B', 'GenePert': '#87A8D8', 'scGPT': '#DDD3C8', 'Perturbed mean': '#B9AEDC', 'Matching mean': '#8D7BC8', 'Truth': '#7F7F7F'}
print('Selected dataset:', DATASET)
print('Selected splits:', SPLIT_IDS)
print('Selected models:', MODELS)
print('Export Fig4c diagnostic panel:', EXPORT_FIG4C)


Selected dataset: norman
Selected splits: [1, 2, 3, 4, 5]
Selected models: ['trishift_nearest', 'biolord', 'gears', 'genepert', 'scgpt', 'systema_nonctl_mean', 'systema_matching_mean']
Export Fig4c diagnostic panel: False


In [3]:
benchmark_dir = OUT_ROOT / 'benchmark'
subprocess.run([
    sys.executable,
    '-m',
    'scripts.trishift.analysis.baseline_panel',
    '--dataset',
    DATASET,
    '--models',
    ','.join(MODELS),
    '--split_ids',
    ','.join(map(str, SPLIT_IDS)),
    '--out_root',
    str(benchmark_dir),
], cwd=repo_root, check=True, capture_output=True, text=True)
summary_df = pd.read_csv(benchmark_dir / 'baseline_panel_summary.csv') if (benchmark_dir / 'baseline_panel_summary.csv').exists() else pd.DataFrame()
subgroup_df = pd.read_csv(benchmark_dir / 'norman_subgroup_panel.csv') if (benchmark_dir / 'norman_subgroup_panel.csv').exists() else pd.DataFrame()
if not summary_df.empty:
    summary_df['label'] = summary_df['model_name'].map(MODEL_LABELS).fillna(summary_df['label'])
if not subgroup_df.empty:
    subgroup_df['label'] = subgroup_df['model_name'].map(MODEL_LABELS).fillna(subgroup_df['label'])
summary_df.to_csv(OUT_ROOT / 'fig4_summary.csv', index=False, encoding='utf-8-sig')
subgroup_df.to_csv(OUT_ROOT / 'fig4_subgroup_summary.csv', index=False, encoding='utf-8-sig')
display(summary_df)
display(subgroup_df.head())


,dataset,model_name,label,n_rows,split_ids_used,metrics_path,mean_pearson,mean_nmse,mean_deg_mean_r2,mean_systema_corr_20de_allpert,mean_systema_corr_deg_r2,mean_scpram_r2_degs_mean_mean,mean_scpram_r2_degs_var_mean,mean_scpram_wasserstein_degs_sum
0,norman,trishift_nearest,TriShift,230,"1,2,3,4,5",E:\CODE\trishift\artifacts\results\norman\metr...,0.872425,0.238160,0.693888,0.777171,0.532452,0.960521,0.533888,5.560348
1,norman,biolord,biolord,267,"1,2,3,4,5",E:\CODE\trishift\artifacts\results\biolord\nor...,0.867309,0.288666,0.647541,0.690994,0.471385,NaN,NaN,NaN
2,norman,gears,GEARS,169,"1,2,3,4,5",E:\CODE\trishift\artifacts\results\gears\norma...,0.775065,0.438067,0.473597,0.543999,0.168901,0.925338,NaN,10.427277
3,norman,genepert,GenePert,230,"1,2,3,4,5",E:\CODE\trishift\artifacts\results\genepert\no...,0.863745,0.370423,0.551701,0.746577,0.293380,0.934912,NaN,10.602059
4,norman,scgpt,scGPT,230,"1,2,3,4,5",E:\CODE\trishift\artifacts\results\scgpt\norma...,0.783110,0.491155,0.383774,0.529376,0.043849,0.901935,0.298204,10.535203
5,norman,systema_nonctl_mean,Perturbed mean,230,"1,2,3,4,5",E:\CODE\trishift\artifacts\results\norman\syst...,0.642935,0.713838,0.092589,-0.288134,-0.335972,NaN,NaN,NaN
6,norman,systema_matching_mean,Matching mean,230,"1,2,3,4,5",E:\CODE\trishift\artifacts\results\norman\syst...,0.765217,0.524235,0.325572,0.308258,-0.000168,NaN,NaN,NaN


,dataset,model_name,label,subgroup,n_rows,mean_pearson,mean_nmse,mean_deg_mean_r2,mean_systema_corr_20de_allpert,mean_systema_corr_deg_r2,mean_scpram_r2_degs_mean_mean,mean_scpram_r2_degs_var_mean,mean_scpram_wasserstein_degs_sum
0,norman,trishift_nearest,TriShift,seen0,6,0.944359,0.130815,0.849167,0.917073,0.791849,0.936267,0.390632,6.130675
1,norman,trishift_nearest,TriShift,seen1,48,0.962918,0.098010,0.875014,0.929692,0.794450,0.966424,0.487207,5.598991
2,norman,trishift_nearest,TriShift,seen2,71,0.973237,0.097522,0.885357,0.946386,0.801829,0.968786,0.521676,5.659639
3,norman,trishift_nearest,TriShift,single,105,0.758777,0.403461,0.472745,0.585030,0.215709,0.953619,0.571672,5.442953
4,norman,biolord,biolord,seen0,6,0.934401,0.209221,0.715126,0.826611,0.515808,NaN,NaN,NaN


In [4]:
try:
    metrics_df = load_metrics_df(resolve_result(dataset=DATASET, model_name='trishift_nearest'))
    schema_df = metrics_df[['condition', 'subgroup']].drop_duplicates().groupby('subgroup', as_index=False).size().rename(columns={'size': 'n_conditions'})
except Exception:
    schema_df = pd.DataFrame(columns=['subgroup', 'n_conditions'])

schema_df.to_csv(OUT_ROOT / 'fig4a_subgroup_schema.csv', index=False, encoding='utf-8-sig')
fig, ax = plt.subplots(figsize=(6.3, 4.0), dpi=220)
if schema_df.empty:
    ax.text(0.5, 0.5, 'Norman subgroup counts unavailable', ha='center', va='center'); ax.axis('off')
else:
    sns.barplot(data=schema_df, x='subgroup', y='n_conditions', order=['single', 'seen2', 'seen1', 'seen0'], color='#4C78A8', ax=ax)
    ax.set_xlabel(''); ax.set_ylabel('Number of conditions'); ax.set_title('Norman subgroup split')
    style_axis(ax, grid_axis='y')
    for patch in ax.patches:
        height = patch.get_height(); ax.text(patch.get_x() + patch.get_width() / 2, height, f'{int(height)}', ha='center', va='bottom', fontsize=9)
fig.tight_layout(); fig.savefig(OUT_ROOT / 'fig4a_subgroup_schema.png', bbox_inches='tight'); plt.close(fig)


In [5]:
def _shrink_bars(ax, scale=0.9):
    for patch in ax.patches:
        width = patch.get_width()
        if width <= 0:
            continue
        new_width = width * scale
        patch.set_x(patch.get_x() + (width - new_width) / 2)
        patch.set_width(new_width)


metrics_to_plot = [
    ('pearson', 'Pearson', 'fig4b_subgroup_pearson.png'),
    ('nmse', 'NMSE', 'fig4c_subgroup_nmse.png'),
    ('deg_mean_r2', 'DEG mean R2', 'fig4d_subgroup_deg_mean_r2.png'),
    ('systema_corr_20de_allpert', 'Systema Pearson', 'fig4e_subgroup_systema_pearson.png'),
]
plot_df = subgroup_df[subgroup_df['subgroup'].isin(['seen2', 'seen1', 'seen0'])].copy()
for metric, metric_label, out_name in metrics_to_plot:
    fig, ax = plt.subplots(1, 1, figsize=(6.5, 4.9), dpi=220)
    if plot_df.empty or f'mean_{metric}' not in plot_df.columns:
        ax.text(0.5, 0.5, f'No rows for {metric}', ha='center', va='center'); ax.axis('off')
    else:
        sns.barplot(data=plot_df, x='subgroup', y=f'mean_{metric}', hue='label', order=['seen2', 'seen1', 'seen0'], palette=MODEL_COLORS, ax=ax, errorbar=None, saturation=0.95)
        _shrink_bars(ax, scale=0.9)
        ax.set_xlabel('')
        ax.set_ylabel(metric_label)
        for patch in ax.patches:
            patch.set_edgecolor('black'); patch.set_linewidth(0.5)
        ax.tick_params(axis='x', rotation=20)
        handles, labels = ax.get_legend_handles_labels()
        if ax.legend_ is not None:
            ax.legend_.remove()
        fig.legend(handles, labels, frameon=False, title='', ncol=3, loc='upper left', bbox_to_anchor=(0.03, 0.995), borderaxespad=0.0, handlelength=0.9, handletextpad=0.35, columnspacing=0.8)
        fig.suptitle(metric_label, y=0.91)
    fig.tight_layout(rect=[0.0, 0.0, 1.0, 0.82])
    fig.savefig(OUT_ROOT / out_name, bbox_inches='tight')
    plt.close(fig)


In [6]:
focus_rows = []
if not plot_df.empty:
    for subgroup in ['seen1', 'seen0']:
        sub = plot_df[plot_df['subgroup'] == subgroup].copy()
        tri = sub[sub['model_name'] == 'trishift_nearest']
        competitors = sub[sub['model_name'] != 'trishift_nearest']
        if tri.empty or competitors.empty:
            continue
        focus_rows.append({'subgroup': subgroup, 'pearson_gain_vs_best': float(tri['mean_pearson'].iloc[0]) - float(competitors['mean_pearson'].max()), 'deg_mean_r2_gain_vs_best': float(tri['mean_deg_mean_r2'].iloc[0]) - float(competitors['mean_deg_mean_r2'].max())})
focus_df = pd.DataFrame(focus_rows)
if EXPORT_FIG4C:
    focus_df.to_csv(OUT_ROOT / 'figs4x_hard_regime_focus.csv', index=False, encoding='utf-8-sig')
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.0), dpi=220)
    if focus_df.empty:
        for ax in axes:
            ax.text(0.5, 0.5, 'No hard-regime summary available', ha='center', va='center'); ax.axis('off')
    else:
        sns.barplot(data=focus_df, x='subgroup', y='pearson_gain_vs_best', color='#4C78A8', ax=axes[0]); axes[0].set_title('Pearson gain over best baseline'); axes[0].set_xlabel('')
        sns.barplot(data=focus_df, x='subgroup', y='deg_mean_r2_gain_vs_best', color='#54A24B', ax=axes[1]); axes[1].set_title('DEG mean R2 gain over best baseline'); axes[1].set_xlabel('')
    fig.tight_layout(); fig.savefig(OUT_ROOT / 'figs4x_hard_regime_focus.png', bbox_inches='tight'); plt.close(fig)
    display(focus_df)
else:
    print('Fig4 hard-regime focus export is disabled by configuration.')


Fig4 hard-regime focus export is disabled by configuration.


In [7]:
per_model = []
metric_load_errors = []
for model_name in MODELS:
    try:
        resolved = resolve_result(dataset=DATASET, model_name=model_name)
        model_df = load_metrics_df(resolved).assign(model_name=model_name)
    except Exception as exc:
        metric_load_errors.append(f'{model_name}: {exc}')
    else:
        per_model.append(model_df)
if metric_load_errors:
    raise RuntimeError('Failed to load Fig4 metrics for one or more models:\n' + '\n'.join(metric_load_errors))
all_metrics = pd.concat(per_model, ignore_index=True) if per_model else pd.DataFrame()
CASE_SPEC = {'condition': 'CBL+TGFBR2', 'split_id': 4, 'subgroup': 'seen1'}
FIG4F_MODELS = ['trishift_nearest', 'gears', 'genepert', 'scgpt']
case_meta, case_plot_df = None, None

_systema_ctx_cache = {}
def _dense_2d(x):
    if hasattr(x, 'toarray'):
        arr = x.toarray()
    else:
        arr = np.asarray(x)
    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
    return arr

def _get_systema_ctx(split_id: int):
    split_id = int(split_id)
    if split_id in _systema_ctx_cache:
        return _systema_ctx_cache[split_id]
    paths_cfg = systema_core.load_yaml(str(repo_root / 'configs' / 'paths.yaml'))
    h5ad_path = str((repo_root / paths_cfg['datasets'][DATASET]).resolve())
    emb_key = systema_core.DATASET_CONFIG[DATASET]['emb_key']
    emb_path = str((repo_root / paths_cfg['embeddings'][emb_key]).resolve())
    adata = systema_core.load_adata(h5ad_path)
    adata.uns = {}
    embd_df = systema_core.load_embedding_df(emb_path)
    embd_df = systema_core.apply_alias_mapping(embd_df, DATASET)
    degs_cache_path = repo_root / 'artifacts' / 'cache' / 'degs' / f'{DATASET}_degs.pkl'
    degs_cache = systema_core._load_degs_cache(degs_cache_path)
    if degs_cache:
        adata.uns.update(degs_cache)
    data = systema_core.TriShiftData(adata, embd_df)
    data.setup_embedding_index()
    data.build_or_load_degs()
    split_dict = split_by_dataset_policy(data, dataset_name=DATASET, seed=split_id)
    train_adata = split_dict['train']
    train_ctrl_adata = systema_core._ctrl_pool_from_split(data, train_adata)
    mu_pert = systema_core._systema_nonctl_mean_mu(train_adata)
    train_cond_set = set(train_adata.obs[data.label_key].astype(str).unique().tolist())
    ctx = {
        'data': data,
        'train_adata': train_adata,
        'train_ctrl_adata': train_ctrl_adata,
        'mu_pert': mu_pert,
        'train_cond_set': train_cond_set,
        'centroid_cache': {},
        'cond_series': data.adata_all.obs[data.label_key].astype(str).values,
        'var_names': np.asarray(data.adata_all.var_names).astype(str),
        'X_all': data.adata_all.X,
    }
    _systema_ctx_cache[split_id] = ctx
    return ctx

def _systema_matching_payload(split_id: int, condition: str):
    cond = normalize_condition(str(condition))
    ctx = _get_systema_ctx(split_id)
    data = ctx['data']
    deg_idx = np.asarray(systema_core._get_deg_idx(data, cond), dtype=int)
    if deg_idx.size == 0:
        return None
    truth_mask = ctx['cond_series'] == cond
    if not np.any(truth_mask):
        return None
    pred_mean = systema_core._systema_matching_mean_mu(
        condition=cond,
        train_adata=ctx['train_adata'],
        train_cond_set=ctx['train_cond_set'],
        mu_pert=ctx['mu_pert'],
        centroid_cache=ctx['centroid_cache'],
    )
    truth = _dense_2d(ctx['X_all'][truth_mask][:, deg_idx])
    ctrl = _dense_2d(ctx['train_ctrl_adata'][:, deg_idx].X)
    pred = np.asarray(pred_mean[deg_idx], dtype=np.float32).reshape(1, -1)
    genes = ctx['var_names'][deg_idx]
    return {'Pred': pred, 'Ctrl': ctrl, 'Truth': truth, 'DE_name': genes}

def _systema_nonctl_payload(split_id: int, condition: str):
    cond = normalize_condition(str(condition))
    ctx = _get_systema_ctx(split_id)
    data = ctx['data']
    deg_idx = np.asarray(systema_core._get_deg_idx(data, cond), dtype=int)
    if deg_idx.size == 0:
        return None
    truth_mask = ctx['cond_series'] == cond
    if not np.any(truth_mask):
        return None
    truth = _dense_2d(ctx['X_all'][truth_mask][:, deg_idx])
    ctrl = _dense_2d(ctx['train_ctrl_adata'][:, deg_idx].X)
    pred = np.asarray(ctx['mu_pert'][deg_idx], dtype=np.float32).reshape(1, -1)
    genes = ctx['var_names'][deg_idx]
    return {'Pred': pred, 'Ctrl': ctrl, 'Truth': truth, 'DE_name': genes}

if not all_metrics.empty:
    condition_key = normalize_condition(CASE_SPEC['condition'])
    split_id = int(CASE_SPEC['split_id'])
    subgroup = str(CASE_SPEC['subgroup'])
    tri_rows = all_metrics[
        (all_metrics['model_name'].astype(str) == 'trishift_nearest')
        & (all_metrics['split_id'].astype(int) == split_id)
        & (all_metrics['condition'].astype(str).map(normalize_condition) == condition_key)
        & (all_metrics['subgroup'].astype(str) == subgroup)
    ].copy()
    if not tri_rows.empty:
        loaded = {}
        case_load_errors = []
        for model_name in FIG4F_MODELS:
            try:
                if model_name == 'systema_matching_mean':
                    item = _systema_matching_payload(split_id=split_id, condition=condition_key)
                elif model_name == 'systema_nonctl_mean':
                    item = _systema_nonctl_payload(split_id=split_id, condition=condition_key)
                else:
                    _, payload = load_payload_item(dataset=DATASET, model_name=model_name, split_id=split_id, condition=None)
                    item = {normalize_condition(k): v for k, v in payload.items()}.get(condition_key)
            except Exception as exc:
                case_load_errors.append(f'{model_name}: {exc}')
            else:
                if item is None:
                    case_load_errors.append(f'{model_name}: missing payload for {condition_key} split {split_id}')
                else:
                    loaded[model_name] = item
        if case_load_errors:
            raise RuntimeError('Failed to resolve Fig4f case payloads:\n' + '\n'.join(case_load_errors))
        if len(loaded) == len(FIG4F_MODELS) and 'trishift_nearest' in loaded:
            base_item = loaded['trishift_nearest']
            truth = np.asarray(base_item['Truth'], dtype=float)
            ctrl = np.asarray(base_item['Ctrl'], dtype=float)
            truth_delta = truth.mean(axis=0) - ctrl.mean(axis=0)
            genes = np.asarray(base_item.get('DE_name', [f'g{i}' for i in range(truth_delta.shape[0])])).astype(str)
            order = np.argsort(-np.abs(truth_delta))[:12]
            truth_top = truth_delta[order]
            rows = [pd.DataFrame({'Gene': genes[order], 'Expression': truth_top, 'Group': 'Truth'})]
            for model_name in FIG4F_MODELS:
                pred_delta = np.asarray(loaded[model_name]['Pred'], dtype=float).mean(axis=0) - np.asarray(loaded[model_name]['Ctrl'], dtype=float).mean(axis=0)
                rows.append(pd.DataFrame({'Gene': genes[order], 'Expression': pred_delta[order], 'Group': MODEL_LABELS[model_name]}))
            case_meta = {'condition': condition_key, 'split_id': split_id, 'subgroup': subgroup}
            case_plot_df = pd.concat(rows, ignore_index=True)
if case_plot_df is not None:
    case_plot_df.to_csv(OUT_ROOT / 'fig4f_combo_case_values.csv', index=False, encoding='utf-8-sig')
    plt.figure(figsize=(15.2, 5.8), dpi=220)
    ax = sns.barplot(data=case_plot_df, x='Gene', y='Expression', hue='Group', palette=MODEL_COLORS, errorbar=None)
    for patch in ax.patches:
        patch.set_edgecolor('black'); patch.set_linewidth(0.4)
    ax.set_xlabel(''); ax.set_ylabel('Change over control')
    ax.set_title(f"Representative hard-generalization case | {case_meta['subgroup']} | {case_meta['condition']}")
    ax.tick_params(axis='x', rotation=45); ax.legend(title='', frameon=False, ncol=4)
    plt.tight_layout(); plt.savefig(OUT_ROOT / 'fig4f_combo_case.png', bbox_inches='tight'); plt.close()
    print(case_meta); display(case_plot_df.head(20))
else:
    raise RuntimeError(f'No data available for the configured Norman hard-generalization case: {CASE_SPEC}')
print(OUT_ROOT)


{'condition': 'CBL+TGFBR2', 'split_id': 4, 'subgroup': 'seen1'}


,Gene,Expression,Group
0,HBZ,1.988604,Truth
1,HBG2,1.084225,Truth
2,ALAS2,0.991693,Truth
3,CD99,0.916867,Truth
4,HBG1,0.780497,Truth
5,GYPB,0.699856,Truth
6,HBA1,0.647308,Truth
7,RP11-301G19.1,-0.562743,Truth
8,GYPA,0.529526,Truth
9,BLVRB,0.505242,Truth


E:\CODE\trishift\artifacts\paper_figures\main\Fig4_NormanGeneralization
